# Mô hình dò onset cho humming (HumTrans)

Pipeline nhận nốt hiện tại cắt nốt ở chỗ lặng, chỗ nhảy cao độ hoặc chỗ năng lượng tụt. Cách này hay **gộp nhầm hai nốt
cùng cao độ hát liền hơi** và **chẻ nhầm chỗ chỉ hụt hơi**. Notebook này học một mô hình nhỏ (MLP numpy) đoán khung nào là
điểm bắt đầu nốt (onset), từ dataset HumTrans (người ngân + file MIDI nhãn).

Đặc trưng mỗi khung 10 ms: cao độ (cent), xác suất có giọng, độ thay đổi cao độ, năng lượng RMS và độ thay đổi, spectral flux,
onset strength, ghép thêm ±3 khung lân cận. Mô hình xuất ra JSON, backend chạy bằng numpy (không cần GPU hay sklearn).

Không cần GPU. Nên chọn runtime có nhiều CPU nếu có (bước pYIN chạy song song).

## 1. Mã nguồn và thư viện

In [ ]:
REPO_URL = 'https://github.com/ThinhNoBaka/AIComposer.git'
!git clone -q $REPO_URL /content/aicomposer || (cd /content/aicomposer && git pull -q)
%cd /content/aicomposer
%pip install -q -r ml/requirements.txt huggingface_hub
import os
print('Số CPU:', os.cpu_count())

In [ ]:
# Lưu cache đặc trưng lên Drive để lần sau không phải chạy lại pYIN (tuỳ chọn)
from google.colab import drive
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/aicomposer_humming_cache'

## 2. Chạy thử trên dữ liệu giả lập

Kiểm tra mọi thứ chạy được trước khi tải dataset thật (khoảng 1 phút).

In [ ]:
!python ml/humming/train_onset.py --synthetic 40 --out /content/onset_synthetic.json

## 3. Tải HumTrans từ Hugging Face

HumTrans có khoảng 14.000 đoạn ngân (56 giờ) kèm MIDI. Nếu tên repo dưới đây đã đổi, tìm "HumTrans" trên huggingface.co/datasets
và sửa `HF_REPO`. Dataset dạng zip thì ô sau tự giải nén.

In [ ]:
HF_REPO = 'dadinghh2/HumTrans'
DATA = '/content/HumTrans'
from huggingface_hub import snapshot_download
snapshot_download(repo_id=HF_REPO, repo_type='dataset', local_dir=DATA)
!cd $DATA && for z in $(find . -name "*.zip"); do unzip -q -n "$z" -d "$(dirname "$z")"; done
!echo "wav: $(find $DATA -name '*.wav' | wc -l)  mid: $(find $DATA -name '*.mid' | wc -l)"
!find $DATA -maxdepth 2 -name "*.json" | head

## 4. Huấn luyện

- `--limit`: số file dùng (bắt đầu 2000–3000; toàn bộ dataset mất vài giờ cho bước pYIN).
- `--jobs`: số tiến trình song song.
- `--keys`: file JSON chia train/valid/test của HumTrans nếu có (tên file xem ở ô trên), để tập kiểm định đúng như bài báo.
- `--cache-dir`: lưu đặc trưng từng file, chạy lại chỉ mất vài giây.

Script in ra F1 của onset và **note F1 (mir_eval, onset ±50 ms, cao độ ±50 cent) của pipeline hiện tại (`transcribe_array`) so với pipeline có chèn mô hình**
trên tập kiểm định. Chỉ dùng mô hình nếu note F1 tăng.

In [ ]:
import glob
keys = (glob.glob(f'{DATA}/**/*key*.json', recursive=True) or [''])[0]
keys_arg = f'--keys "{keys}"' if keys else ''
print('File chia tập:', keys or '(không có, chia ngẫu nhiên 80/20)')
!python ml/humming/train_onset.py --data $DATA $keys_arg --limit 3000 --jobs {os.cpu_count()} --cache-dir "$CACHE" --epochs 20 --eval-limit 300 --out /content/humming_onset.json

## 5. Xem kỹ hơn kết quả đánh giá

Số liệu script đã ghi vào file, rồi chấm lại trên tập test riêng bằng đúng pipeline của app (`transcribe_array`).

In [ ]:
import json, sys
sys.path.insert(0, 'ml/humming')
m = json.load(open('/content/humming_onset.json'))
print(json.dumps(m['metrics'], indent=1, ensure_ascii=False))
print('Ngưỡng:', m['threshold'], '| số đặc trưng:', len(m['feature_names']))

In [ ]:
# Đánh giá thêm trên phần test riêng (nếu file chia tập có 'test'). Chạy đúng pipeline của app (transcribe_array),
# một lần như hiện tại và một lần có chèn mô hình onset sau bước tách nốt; chấm note F1 bằng mir_eval.
import json
from pathlib import Path
import train_onset as T
from app.humming import onset_model as om
from scripts.eval_humtrans import find_pairs

model = om.OnsetModel.from_dict(m)
pairs = find_pairs(Path(DATA), None)
if keys:
    test_names = {Path(k).stem for k in json.load(open(keys)).get('test', [])}
    pairs = [p for p in pairs if p[0].stem in test_names] or pairs
items = [it for it in (T._process_file((str(a), str(mid), m['context'], 2, CACHE, True)) for a, mid in pairs[:200]) if it]
res = T.evaluate(model, items, 200)
print(f"{res['files']} file test | note F1 pipeline hiện tại {res['note_f1_baseline']:.3f} → có mô hình {res['note_f1_model']:.3f}")

## 6. Tải mô hình về

Chép `humming_onset.json` vào `backend/models/humming_onset.json` trong repo rồi commit. Backend (và image Docker,
`HUMMING_MODEL=/app/models/humming_onset.json`) tự nạp khi khởi động; xoá file thì quay về cách tách nốt cũ.

In [ ]:
from google.colab import files
files.download('/content/humming_onset.json')